In [1]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {
    "id": "start"
   },
   "source": [
    "# Mini-projet :Planification robuste sur grille \n",
    "## A* + Chaînes de Markov\n",
    "\n",
    "**Auteur** : BALMIR SALMA \n",
    "\n",
    "**Date** : 05/03/2026 \n",
    "\n",
    "---\n",
    "\n",
    "## Objectif du notebook\n",
    "\n",
    "Ce notebook présente l'implémentation et l'analyse complète du projet de planification robuste sur grille. Nous allons :\n",
    "\n",
    "1. Implémenter une grille 2D avec obstacles\n",
    "2. Planifier un chemin avec A* et ses variantes\n",
    "3. Modéliser l'incertitude par une chaîne de Markov\n",
    "4. Analyser les propriétés de la chaîne (classes, absorption)\n",
    "5. Simuler des trajectoires Monte-Carlo\n",
    "6. Comparer théorie et pratique\n",
    "7. Réaliser les 4 expériences demandées"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {
    "id": "imports"
   },
   "source": [
    "# -*- coding: utf-8 -*-\n",
    "# Import des modules nécessaires\n",
    "import numpy as np\n",
    "import matplotlib.pyplot as plt\n",
    "import networkx as nx\n",
    "import time\n",
    "from collections import defaultdict\n",
    "import heapq\n",
    "\n",
    "# Configuration de l'affichage\n",
    "%matplotlib inline\n",
    "plt.style.use('ggplot')\n",
    "plt.rcParams['figure.figsize'] = (10, 6)\n",
    "plt.rcParams['font.size'] = 12\n",
    "\n",
    "print(\"✅ Modules importés avec succès\")\n",
    "print(f\"NumPy version: {np.__version__}\")\n",
    "print(f\"Matplotlib version: {plt.matplotlib.__version__}\")\n",
    "print(f\"NetworkX version: {nx.__version__}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {
    "id": "part1"
   },
   "source": [
    "## Partie 1 : Définition de la grille\n",
    "\n",
    "Nous créons une classe `Grid` pour représenter l'environnement."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {
    "id": "grid_class"
   },
   "source": [
    "class Grid:\n",
    "    \"\"\"Grille 2D avec obstacles\"\"\"\n",
    "    \n",
    "    def __init__(self, width, height, obstacles=None):\n",
    "        self.width = width\n",
    "        self.height = height\n",
    "        self.obstacles = set(obstacles) if obstacles else set()\n",
    "        \n",
    "    def is_free(self, x, y):\n",
    "        \"\"\"Vérifie si une cellule est libre\"\"\"\n",
    "        return (0 <= x < self.width and 0 <= y < self.height \n",
    "                and (x, y) not in self.obstacles)\n",
    "    \n",
    "    def neighbors(self, x, y):\n",
    "        \"\"\"Retourne les voisins accessibles (4-connectivité)\"\"\"\n",
    "        candidates = [(x+1, y), (x-1, y), (x, y+1), (x, y-1)]\n",
    "        return [(nx, ny) for nx, ny in candidates if self.is_free(nx, ny)]\n",
    "    \n",
    "    def manhattan_distance(self, pos1, pos2):\n",
    "        \"\"\"Heuristique Manhattan\"\"\"\n",
    "        return abs(pos1[0] - pos2[0]) + abs(pos1[1] - pos2[1])\n",
    "    \n",
    "    def euclidean_distance(self, pos1, pos2):\n",
    "        \"\"\"Heuristique euclidienne\"\"\"\n",
    "        return np.sqrt((pos1[0] - pos2[0])**2 + (pos1[1] - pos2[1])**2)\n",
    "    \n",
    "    def zero_heuristic(self, pos1, pos2):\n",
    "        \"\"\"Heuristique nulle (pour UCS)\"\"\"\n",
    "        return 0\n",
    "    \n",
    "    def plot(self, path=None, title=\"Grille\"):\n",
    "        \"\"\"Visualise la grille et optionnellement un chemin\"\"\"\n",
    "        fig, ax = plt.subplots(figsize=(8, 8))\n",
    "        \n",
    "        # Dessiner les cellules\n",
    "        for x in range(self.width):\n",
    "            for y in range(self.height):\n",
    "                if (x, y) in self.obstacles:\n",
    "                    ax.add_patch(plt.Rectangle((x, y), 1, 1, color='red', alpha=0.7))\n",
    "                else:\n",
    "                    ax.add_patch(plt.Rectangle((x, y), 1, 1, fill=False, edgecolor='gray'))\n",
    "        \n",
    "        # Dessiner le chemin si fourni\n",
    "        if path:\n",
    "            path_x = [p[0] + 0.5 for p in path]\n",
    "            path_y = [p[1] + 0.5 for p in path]\n",
    "            ax.plot(path_x, path_y, 'b-', linewidth=3, label='Chemin')\n",
    "            ax.scatter(path_x[0], path_y[0], color='green', s=200, label='Départ', zorder=5)\n",
    "            ax.scatter(path_x[-1], path_y[-1], color='blue', s=200, label='But', zorder=5)\n",
    "        \n",
    "        ax.set_xlim(0, self.width)\n",
    "        ax.set_ylim(0, self.height)\n",
    "        ax.set_aspect('equal')\n",
    "        ax.set_xticks(range(self.width))\n",
    "        ax.set_yticks(range(self.height))\n",
    "        ax.grid(True, alpha=0.3)\n",
    "        ax.legend()\n",
    "        ax.set_title(title)\n",
    "        plt.show()\n",
    "        \n",
    "        return fig"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {
    "id": "test_grid"
   },
   "source": [
    "### Test de la grille"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {
    "id": "create_test_grid"
   },
   "source": [
    "# Créer une grille de test\n",
    "grid = Grid(8, 8, obstacles=[(2,2), (2,3), (3,2), (5,5), (5,6), (6,5)])\n",
    "start = (0, 0)\n",
    "goal = (7, 7)\n",
    "\n",
    "print(f\"Grille: {grid.width}x{grid.height}\")\n",
    "print(f\"Obstacles: {grid.obstacles}\")\n",
    "print(f\"Départ: {start}, But: {goal}\")\n",
    "\n",
    "grid.plot(title=\"Grille de test\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {
    "id": "part2"
   },
   "source": [
    "## Partie 2 : Algorithme A* et variantes\n",
    "\n",
    "Implémentation de A*, UCS, Greedy et Weighted A*."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {
    "id": "astar_class"
   },
   "source": [
    "class AStar:\n",
    "    \"\"\"Implémentation de A* et variantes\"\"\"\n",
    "    \n",
    "    def __init__(self, grid, heuristic='manhattan'):\n",
    "        self.grid = grid\n",
    "        self.heuristic_map = {\n",
    "            'manhattan': grid.manhattan_distance,\n",
    "            'euclidean': grid.euclidean_distance,\n",
    "            'zero': grid.zero_heuristic\n",
    "        }\n",
    "        self.heuristic = self.heuristic_map.get(heuristic, grid.manhattan_distance)\n",
    "        \n",
    "    def search(self, start, goal, weight=1.0, algorithm='astar'):\n",
    "        \"\"\"\n",
    "        Recherche du chemin optimal\n",
    "        algorithm: 'astar', 'ucs', 'greedy'\n",
    "        weight: facteur de pondération pour A* (weighted A*)\n",
    "        \"\"\"\n",
    "        # Adaptation des poids selon l'algorithme\n",
    "        if algorithm == 'ucs':\n",
    "            weight_g = 1.0\n",
    "            weight_h = 0.0\n",
    "        elif algorithm == 'greedy':\n",
    "            weight_g = 0.0\n",
    "            weight_h = 1.0\n",
    "        else:  # A* standard ou pondéré\n",
    "            weight_g = 1.0\n",
    "            weight_h = weight\n",
    "        \n",
    "        counter = 0\n",
    "        open_set = []\n",
    "        heapq.heappush(open_set, (0, counter, start))\n",
    "        \n",
    "        came_from = {}\n",
    "        g_score = {start: 0}\n",
    "        \n",
    "        open_set_lookup = {start}\n",
    "        closed_set = set()\n",
    "        \n",
    "        nodes_expanded = 0\n",
    "        max_open_size = 0\n",
    "        \n",
    "        while open_set:\n",
    "            max_open_size = max(max_open_size, len(open_set))\n",
    "            current_f, _, current = heapq.heappop(open_set)\n",
    "            open_set_lookup.remove(current)\n",
    "            \n",
    "            if current == goal:\n",
    "                path = self.reconstruct_path(came_from, current)\n",
    "                return {\n",
    "                    'path': path,\n",
    "                    'cost': g_score[current],\n",
    "                    'nodes_expanded': nodes_expanded,\n",
    "                    'max_open_size': max_open_size,\n",
    "                    'success': True\n",
    "                }\n",
    "            \n",
    "            closed_set.add(current)\n",
    "            nodes_expanded += 1\n",
    "            \n",
    "            for neighbor in self.grid.neighbors(*current):\n",
    "                if neighbor in closed_set:\n",
    "                    continue\n",
    "                \n",
    "                tentative_g = g_score[current] + 1\n",
    "                \n",
    "                if neighbor not in g_score or tentative_g < g_score[neighbor]:\n",
    "                    came_from[neighbor] = current\n",
    "                    g_score[neighbor] = tentative_g\n",
    "                    f = weight_g * tentative_g + weight_h * self.heuristic(neighbor, goal)\n",
    "                    \n",
    "                    if neighbor not in open_set_lookup:\n",
    "                        counter += 1\n",
    "                        heapq.heappush(open_set, (f, counter, neighbor))\n",
    "                        open_set_lookup.add(neighbor)\n",
    "        \n",
    "        return {\n",
    "            'path': None,\n",
    "            'cost': float('inf'),\n",
    "            'nodes_expanded': nodes_expanded,\n",
    "            'max_open_size': max_open_size,\n",
    "            'success': False\n",
    "        }\n",
    "    \n",
    "    def reconstruct_path(self, came_from, current):\n",
    "        \"\"\"Reconstruit le chemin du début à la fin\"\"\"\n",
    "        path = [current]\n",
    "        while current in came_from:\n",
    "            current = came_from[current]\n",
    "            path.append(current)\n",
    "        path.reverse()\n",
    "        return path\n",
    "    \n",
    "    def extract_policy(self, path):\n",
    "        \"\"\"Extrait une politique du chemin\"\"\"\n",
    "        policy = {}\n",
    "        for i in range(len(path) - 1):\n",
    "            current = path[i]\n",
    "            next_state = path[i + 1]\n",
    "            action = (next_state[0] - current[0], next_state[1] - current[1])\n",
    "            policy[current] = action\n",
    "        if path:\n",
    "            policy[path[-1]] = (0, 0)\n",
    "        return policy"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {
    "id": "test_astar"
   },
   "source": [
    "### Test de A* sur la grille"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {
    "id": "run_astar"
   },
   "source": [
    "astar = AStar(grid)\n",
    "result = astar.search(start, goal, algorithm='astar')\n",
    "\n",
    "print(f\"Chemin trouvé: {result['success']}\")\n",
    "print(f\"Longueur du chemin: {len(result['path'])}\")\n",
    "print(f\"Coût total: {result['cost']}\")\n",
    "print(f\"Nœuds développés: {result['nodes_expanded']}\")\n",
    "\n",
    "grid.plot(result['path'], title=\"Chemin trouvé par A*\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {
    "id": "part3"
   },
   "source": [
    "## Partie 3 : Chaîne de Markov\n",
    "\n",
    "Modélisation de l'incertitude d'exécution."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {
    "id": "markov_class"
   },
   "source": [
    "class MarkovChain:\n",
    "    \"\"\"Chaîne de Markov à temps discret\"\"\"\n",
    "    \n",
    "    def __init__(self, states, absorbing_states=None):\n",
    "        self.states = list(states)\n",
    "        self.state_to_idx = {state: i for i, state in enumerate(self.states)}\n",
    "        self.idx_to_state = {i: state for i, state in enumerate(self.states)}\n",
    "        self.n = len(states)\n",
    "        self.P = np.zeros((self.n, self.n))\n",
    "        self.absorbing_states = absorbing_states if absorbing_states else {}\n",
    "        self.absorbing_indices = {self.state_to_idx[s] for s in absorbing_states.keys() if s in self.state_to_idx}\n",
    "    \n",
    "    def add_transition(self, from_state, to_state, probability):\n",
    "        if from_state not in self.state_to_idx or to_state not in self.state_to_idx:\n",
    "            return\n",
    "        i = self.state_to_idx[from_state]\n",
    "        j = self.state_to_idx[to_state]\n",
    "        self.P[i, j] += probability\n",
    "    \n",
    "    def normalize_rows(self):\n",
    "        row_sums = self.P.sum(axis=1)\n",
    "        row_sums[row_sums == 0] = 1\n",
    "        self.P = self.P / row_sums[:, np.newaxis]\n",
    "    \n",
    "    def is_stochastic(self):\n",
    "        row_sums = self.P.sum(axis=1)\n",
    "        return np.allclose(row_sums, 1.0)\n",
    "    \n",
    "    def build_from_policy(self, policy, grid, epsilon, fail_states=None):\n",
    "        \"\"\"Construit la matrice P à partir d'une politique\"\"\"\n",
    "        # Identifier l'état but\n",
    "        goal_state = None\n",
    "        for state, action in policy.items():\n",
    "            if action == (0, 0):\n",
    "                goal_state = state\n",
    "                self.absorbing_states[goal_state] = 'GOAL'\n",
    "                break\n",
    "        \n",
    "        # Ajouter FAIL\n",
    "        fail_state = None\n",
    "        if fail_states:\n",
    "            fail_state = (-1, -1)\n",
    "            while fail_state in self.state_to_idx:\n",
    "                fail_state = (fail_state[0] - 1, fail_state[1] - 1)\n",
    "            if fail_state not in self.state_to_idx:\n",
    "                self.states.append(fail_state)\n",
    "                self.state_to_idx[fail_state] = self.n\n",
    "                self.idx_to_state[self.n] = fail_state\n",
    "                self.n += 1\n",
    "                self.absorbing_states[fail_state] = 'FAIL'\n",
    "                self.absorbing_indices.add(self.n - 1)\n",
    "                new_P = np.zeros((self.n, self.n))\n",
    "                new_P[:self.n-1, :self.n-1] = self.P\n",
    "                self.P = new_P\n",
    "        \n",
    "        valid_states = set(self.states)\n",
    "        \n",
    "        # Remplir la matrice\n",
    "        for state in self.states:\n",
    "            if state in self.absorbing_states:\n",
    "                i = self.state_to_idx[state]\n",
    "                self.P[i, i] = 1.0\n",
    "                continue\n",
    "            \n",
    "            if state == fail_state:\n",
    "                continue\n",
    "            \n",
    "            i = self.state_to_idx[state]\n",
    "            \n",
    "            if state in policy:\n",
    "                action = policy[state]\n",
    "                intended_next = (state[0] + action[0], state[1] + action[1])\n",
    "            else:\n",
    "                intended_next = state\n",
    "                action = (0, 0)\n",
    "            \n",
    "            if not grid.is_free(*intended_next) or intended_next not in valid_states:\n",
    "                intended_next = state\n",
    "            \n",
    "            self.add_transition(state, intended_next, 1 - epsilon)\n",
    "            \n",
    "            if action != (0, 0):\n",
    "                lateral_moves = []\n",
    "                if action[0] != 0:\n",
    "                    lateral_moves = [(0, 1), (0, -1)]\n",
    "                elif action[1] != 0:\n",
    "                    lateral_moves = [(1, 0), (-1, 0)]\n",
    "                \n",
    "                for lat_action in lateral_moves:\n",
    "                    lat_next = (state[0] + lat_action[0], state[1] + lat_action[1])\n",
    "                    if not grid.is_free(*lat_next) or lat_next not in valid_states:\n",
    "                        lat_next = state\n",
    "                    self.add_transition(state, lat_next, epsilon / 2)\n",
    "            \n",
    "            if fail_states and state in fail_states and fail_state in valid_states:\n",
    "                self.add_transition(state, fail_state, 1.0)\n",
    "        \n",
    "        self.normalize_rows()\n",
    "    \n",
    "    def get_distribution(self, initial_state, n):\n",
    "        i0 = self.state_to_idx[initial_state]\n",
    "        pi0 = np.zeros(self.n)\n",
    "        pi0[i0] = 1.0\n",
    "        Pn = np.linalg.matrix_power(self.P, n)\n",
    "        return pi0 @ Pn\n",
    "    \n",
    "    def analyze_classes(self):\n",
    "        \"\"\"Identifie les classes de communication\"\"\"\n",
    "        G = nx.DiGraph()\n",
    "        G.add_nodes_from(range(self.n))\n",
    "        for i in range(self.n):\n",
    "            for j in range(self.n):\n",
    "                if self.P[i, j] > 0:\n",
    "                    G.add_edge(i, j)\n",
    "        \n",
    "        try:\n",
    "            cfc = list(nx.strongly_connected_components(G))\n",
    "        except:\n",
    "            cfc = [{i} for i in range(self.n)]\n",
    "        \n",
    "        classes = []\n",
    "        for comp in cfc:\n",
    "            is_closed = True\n",
    "            for i in comp:\n",
    "                for j in range(self.n):\n",
    "                    if self.P[i, j] > 0 and j not in comp:\n",
    "                        is_closed = False\n",
    "                        break\n",
    "                if not is_closed:\n",
    "                    break\n",
    "            \n",
    "            class_type = 'absorbante' if is_closed else 'transitoire'\n",
    "            if len(comp) == 1:\n",
    "                i = next(iter(comp))\n",
    "                if self.P[i, i] == 1.0:\n",
    "                    class_type = 'absorbant'\n",
    "            \n",
    "            classes.append({\n",
    "                'etats': [self.idx_to_state[idx] for idx in comp],\n",
    "                'type': class_type\n",
    "            })\n",
    "        return classes\n",
    "    \n",
    "    def absorption_analysis(self):\n",
    "        \"\"\"Analyse d'absorption\"\"\"\n",
    "        if not self.absorbing_indices:\n",
    "            return None\n",
    "        \n",
    "        trans_indices = [i for i in range(self.n) if i not in self.absorbing_indices]\n",
    "        abs_indices = list(self.absorbing_indices)\n",
    "        \n",
    "        if not trans_indices:\n",
    "            return None\n",
    "        \n",
    "        Q = self.P[np.ix_(trans_indices, trans_indices)]\n",
    "        I = np.eye(len(trans_indices))\n",
    "        \n",
    "        try:\n",
    "            N = np.linalg.inv(I - Q)\n",
    "            R = self.P[np.ix_(trans_indices, abs_indices)]\n",
    "            B = N @ R\n",
    "            t = N @ np.ones(len(trans_indices))\n",
    "            return {'B': B, 't': t, 'trans_indices': trans_indices, 'abs_indices': abs_indices}\n",
    "        except:\n",
    "            return None"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {
    "id": "build_markov"
   },
   "source": [
    "### Construction de la chaîne de Markov à partir du chemin"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {
    "id": "build_chain"
   },
   "source": [
    "# Extraire la politique\n",
    "policy = astar.extract_policy(result['path'])\n",
    "fail_states = grid.obstacles\n",
    "absorbing = {goal: 'GOAL', (-1, -1): 'FAIL'}\n",
    "\n",
    "# Inclure tous les états nécessaires\n",
    "all_states = set(result['path']) | set(fail_states) | {(-1, -1)}\n",
    "for state in result['path']:\n",
    "    for neighbor in grid.neighbors(*state):\n",
    "        all_states.add(neighbor)\n",
    "\n",
    "print(f\"Nombre d'états dans la chaîne: {len(all_states)}\")\n",
    "\n",
    "# Construire avec ε=0.2\n",
    "epsilon = 0.2\n",
    "mc = MarkovChain(all_states, absorbing)\n",
    "mc.build_from_policy(policy, grid, epsilon, fail_states)\n",
    "\n",
    "print(f\"\\nMatrice P (taille {mc.n}x{mc.n}):\")\n",
    "print(mc.P[:6, :6])  # Afficher les 6 premières lignes/colonnes\n",
    "print(f\"\\nStochastique: {mc.is_stochastic()}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {
    "id": "analysis"
   },
   "source": [
    "### Analyse de la chaîne"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {
    "id": "class_analysis"
   },
   "source": [
    "# Analyse des classes\n",
    "classes = mc.analyze_classes()\n",
    "print(\"\\nCLASSES DE COMMUNICATION:\")\n",
    "for i, cls in enumerate(classes):\n",
    "    print(f\"  Classe {i+1} ({cls['type']}): {cls['etats']}\")\n",
    "\n",
    "# Distribution après n étapes\n",
    "print(\"\\nÉVOLUTION DE LA DISTRIBUTION:\")\n",
    "for n in [1, 5, 10, 20]:\n",
    "    dist = mc.get_distribution(start, n)\n",
    "    p_goal = dist[mc.state_to_idx[goal]] if goal in mc.state_to_idx else 0\n",
    "    p_fail = dist[mc.state_to_idx[(-1, -1)]] if (-1, -1) in mc.state_to_idx else 0\n",
    "    print(f\"  n={n}: P(goal)={p_goal:.4f}, P(fail)={p_fail:.4f}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {
    "id": "part4"
   },
   "source": [
    "## Partie 4 : Simulation Monte-Carlo"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {
    "id": "simulation_class"
   },
   "source": [
    "class MarkovSimulation:\n",
    "    \"\"\"Simulation Monte-Carlo\"\"\"\n",
    "    \n",
    "    def __init__(self, markov_chain):\n",
    "        self.mc = markov_chain\n",
    "    \n",
    "    def simulate_trajectory(self, start_state, max_steps=100):\n",
    "        if start_state not in self.mc.state_to_idx:\n",
    "            return None, None, None\n",
    "        \n",
    "        trajectory = [start_state]\n",
    "        current_idx = self.mc.state_to_idx[start_state]\n",
    "        \n",
    "        for step in range(max_steps):\n",
    "            probs = self.mc.P[current_idx]\n",
    "            next_idx = np.random.choice(self.mc.n, p=probs)\n",
    "            next_state = self.mc.idx_to_state[next_idx]\n",
    "            trajectory.append(next_state)\n",
    "            \n",
    "            if next_state in self.mc.absorbing_states:\n",
    "                return trajectory, step + 1, next_state\n",
    "            \n",
    "            current_idx = next_idx\n",
    "        \n",
    "        return trajectory, max_steps, trajectory[-1]\n",
    "    \n",
    "    def run_simulations(self, start_state, n_simulations=1000, max_steps=100):\n",
    "        stats = {\n",
    "            'goal_count': 0, 'fail_count': 0, 'other_count': 0,\n",
    "            'times_to_goal': [], 'times_to_fail': [],\n",
    "            'trajectories': [], 'final_states': defaultdict(int)\n",
    "        }\n",
    "        \n",
    "        for _ in range(n_simulations):\n",
    "            traj, time, final = self.simulate_trajectory(start_state, max_steps)\n",
    "            if final is None:\n",
    "                continue\n",
    "            \n",
    "            stats['trajectories'].append(traj)\n",
    "            stats['final_states'][final] += 1\n",
    "            \n",
    "            if final in self.mc.absorbing_states:\n",
    "                abs_type = self.mc.absorbing_states[final]\n",
    "                if abs_type == 'GOAL':\n",
    "                    stats['goal_count'] += 1\n",
    "                    stats['times_to_goal'].append(time)\n",
    "                elif abs_type == 'FAIL':\n",
    "                    stats['fail_count'] += 1\n",
    "                    stats['times_to_fail'].append(time)\n",
    "            else:\n",
    "                stats['other_count'] += 1\n",
    "        \n",
    "        return stats"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {
    "id": "run_sim"
   },
   "source": [
    "# Lancer les simulations\n",
    "sim = MarkovSimulation(mc)\n",
    "stats = sim.run_simulations(start, n_simulations=1000, max_steps=20)\n",
    "\n",
    "print(\"\\nRÉSULTATS DE SIMULATION:\")\n",
    "print(f\"  P(goal) empirique: {stats['goal_count']/1000:.4f}\")\n",
    "print(f\"  P(fail) empirique: {stats['fail_count']/1000:.4f}\")\n",
    "if stats['times_to_goal']:\n",
    "    print(f\"  Temps moyen vers goal: {np.mean(stats['times_to_goal']):.2f} étapes\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {
    "id": "visualize_trajectories"
   },
   "source": [
    "### Visualisation des trajectoires"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {
    "id": "plot_trajectories"
   },
   "source": [
    "fig, ax = plt.subplots(figsize=(10, 10))\n",
    "\n",
    "# Dessiner la grille\n",
    "for x in range(grid.width):\n",
    "    for y in range(grid.height):\n",
    "        if (x, y) in grid.obstacles:\n",
    "            ax.add_patch(plt.Rectangle((x, y), 1, 1, color='red', alpha=0.3))\n",
    "        else:\n",
    "            ax.add_patch(plt.Rectangle((x, y), 1, 1, fill=False, edgecolor='gray'))\n",
    "\n",
    "# Tracer 30 trajectoires\n",
    "for i in range(min(30, len(stats['trajectories']))):\n",
    "    traj = stats['trajectories'][i]\n",
    "    traj_x = [p[0] + 0.5 for p in traj if isinstance(p, tuple) and len(p) == 2]\n",
    "    traj_y = [p[1] + 0.5 for p in traj if isinstance(p, tuple) and len(p) == 2]\n",
    "    alpha = 0.3 if traj[-1] == goal else 0.1\n",
    "    color = 'blue' if traj[-1] == goal else 'red'\n",
    "    ax.plot(traj_x, traj_y, color=color, alpha=alpha, linewidth=1)\n",
    "\n",
    "ax.scatter([start[0]+0.5], [start[1]+0.5], color='green', s=200, label='Départ', zorder=5)\n",
    "ax.scatter([goal[0]+0.5], [goal[1]+0.5], color='blue', s=200, label='But', zorder=5)\n",
    "ax.set_xlim(0, grid.width)\n",
    "ax.set_ylim(0, grid.height)\n",
    "ax.set_aspect('equal')\n",
    "ax.set_title(f\"30 trajectoires simulées (ε={epsilon})\")\n",
    "ax.legend()\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {
    "id": "part5"
   },
   "source": [
    "## Partie 5 : Expérience 1 - Comparaison des algorithmes"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {
    "id": "exp1"
   },
   "source": [
    "def compare_algorithms_demo():\n",
    "    \"\"\"Compare A*, UCS, Greedy sur la grille actuelle\"\"\"\n",
    "    astar = AStar(grid)\n",
    "    algorithms = ['astar', 'ucs', 'greedy']\n",
    "    results = {}\n",
    "    \n",
    "    for algo in algorithms:\n",
    "        start_time = time.time()\n",
    "        result = astar.search(start, goal, algorithm=algo)\n",
    "        elapsed = time.time() - start_time\n",
    "        results[algo] = {**result, 'time': elapsed}\n",
    "    \n",
    "    # Affichage\n",
    "    print(\"\\n\" + \"=\"*60)\n",
    "    print(\"COMPARAISON DES ALGORITHMES\")\n",
    "    print(\"=\"*60)\n",
    "    \n",
    "    for algo in algorithms:\n",
    "        r = results[algo]\n",
    "        print(f\"\\n{algo.upper()}:\")\n",
    "        print(f\"  Coût: {r['cost']}\")\n",
    "        print(f\"  Nœuds développés: {r['nodes_expanded']}\")\n",
    "        print(f\"  Temps: {r['time']*1000:.3f} ms\")\n",
    "    \n",
    "    # Graphique\n",
    "    fig, ax = plt.subplots(figsize=(8, 5))\n",
    "    x = np.arange(len(algorithms))\n",
    "    nodes = [results[a]['nodes_expanded'] for a in algorithms]\n",
    "    \n",
    "    ax.bar(x, nodes, color=['blue', 'green', 'red'], alpha=0.7)\n",
    "    ax.set_xlabel('Algorithmes')\n",
    "    ax.set_ylabel('Nœuds développés')\n",
    "    ax.set_title('Comparaison du nombre de nœuds développés')\n",
    "    ax.set_xticks(x)\n",
    "    ax.set_xticklabels(algorithms)\n",
    "    ax.grid(True, alpha=0.3)\n",
    "    \n",
    "    for i, v in enumerate(nodes):\n",
    "        ax.text(i, v + 1, str(v), ha='center', va='bottom')\n",
    "    \n",
    "    plt.show()\n",
    "    \n",
    "    return results\n",
    "\n",
    "results_exp1 = compare_algorithms_demo()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {
    "id": "part6"
   },
   "source": [
    "## Partie 6 : Expérience 2 - Impact de l'incertitude ε"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {
    "id": "exp2"
   },
   "source": [
    "def vary_epsilon_demo():\n",
    "    \"\"\"Étudie l'impact de ε sur P(goal)\"\"\"\n",
    "    epsilons = [0.0, 0.1, 0.2, 0.3]\n",
    "    p_goals = []\n",
    "    \n",
    "    for eps in epsilons:\n",
    "        mc_temp = MarkovChain(all_states, absorbing)\n",
    "        mc_temp.build_from_policy(policy, grid, eps, fail_states)\n",
    "        dist = mc_temp.get_distribution(start, 20)\n",
    "        p_goal = dist[mc_temp.state_to_idx[goal]]\n",
    "        p_goals.append(p_goal)\n",
    "    \n",
    "    # Graphique\n",
    "    fig, ax = plt.subplots(figsize=(8, 5))\n",
    "    ax.plot(epsilons, p_goals, 'bo-', linewidth=2, markersize=8)\n",
    "    ax.set_xlabel('Taux d\\'incertitude ε')\n",
    "    ax.set_ylabel('P(atteindre GOAL) après 20 étapes')\n",
    "    ax.set_title('Impact de l\\'incertitude sur la probabilité de succès')\n",
    "    ax.grid(True, alpha=0.3)\n",
    "    ax.set_ylim(0, 1.05)\n",
    "    \n",
    "    for i, (eps, p) in enumerate(zip(epsilons, p_goals)):\n",
    "        ax.text(eps, p + 0.02, f'{p:.3f}', ha='center')\n",
    "    \n",
    "    plt.show()\n",
    "    \n",
    "    return epsilons, p_goals\n",
    "\n",
    "epsilons, p_goals = vary_epsilon_demo()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {
    "id": "part7"
   },
   "source": [
    "## Partie 7 : Expérience 3 - Comparaison des heuristiques"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {
    "id": "exp3"
   },
   "source": [
    "def compare_heuristics_demo():\n",
    "    \"\"\"Compare Manhattan vs heuristique nulle\"\"\"\n",
    "    heuristics = ['manhattan', 'zero']\n",
    "    nodes_count = []\n",
    "    \n",
    "    for heur in heuristics:\n",
    "        astar_h = AStar(grid, heuristic=heur)\n",
    "        result = astar_h.search(start, goal, algorithm='astar')\n",
    "        nodes_count.append(result['nodes_expanded'])\n",
    "    \n",
    "    # Graphique\n",
    "    fig, ax = plt.subplots(figsize=(8, 5))\n",
    "    bars = ax.bar(heuristics, nodes_count, color=['blue', 'orange'], alpha=0.7)\n",
    "    ax.set_xlabel('Heuristique')\n",
    "    ax.set_ylabel('Nœuds développés')\n",
    "    ax.set_title('Comparaison des heuristiques')\n",
    "    ax.grid(True, alpha=0.3)\n",
    "    \n",
    "    for bar, val in zip(bars, nodes_count):\n",
    "        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5, str(val), ha='center')\n",
    "    \n",
    "    reduction = (1 - nodes_count[0]/nodes_count[1]) * 100\n",
    "    print(f\"\\nRéduction du nombre de nœuds: {reduction:.1f}%\")\n",
    "    \n",
    "    plt.show()\n",
    "    \n",
    "    return heuristics, nodes_count\n",
    "\n",
    "heuristics, nodes_count = compare_heuristics_demo()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {
    "id": "part8"
   },
   "source": [
    "## Partie 8 : Expérience 4 - Weighted A*"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {
    "id": "exp4"
   },
   "source": [
    "def weighted_astar_demo():\n",
    "    \"\"\"Étudie le compromis vitesse/optimalité\"\"\"\n",
    "    weights = [1.0, 1.5, 2.0, 3.0, 5.0]\n",
    "    costs = []\n",
    "    nodes = []\n",
    "    \n",
    "    astar_w = AStar(grid)\n",
    "    optimal = astar_w.search(start, goal, algorithm='astar', weight=1.0)\n",
    "    optimal_cost = optimal['cost']\n",
    "    \n",
    "    for w in weights:\n",
    "        result = astar_w.search(start, goal, algorithm='astar', weight=w)\n",
    "        costs.append(result['cost'])\n",
    "        nodes.append(result['nodes_expanded'])\n",
    "    \n",
    "    # Graphique\n",
    "    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))\n",
    "    \n",
    "    # Sous-optimalité\n",
    "    subopt = [(c - optimal_cost)/optimal_cost * 100 for c in costs]\n",
    "    ax1.plot(weights, subopt, 'bo-', linewidth=2, markersize=8)\n",
    "    ax1.set_xlabel('Poids w')\n",
    "    ax1.set_ylabel('Sous-optimalité (%)')\n",
    "    ax1.set_title('Sous-optimalité vs Poids')\n",
    "    ax1.grid(True, alpha=0.3)\n",
    "    \n",
    "    # Accélération\n",
    "    speedup = [nodes[0]/n for n in nodes]\n",
    "    ax2.plot(weights, speedup, 'ro-', linewidth=2, markersize=8)\n",
    "    ax2.set_xlabel('Poids w')\n",
    "    ax2.set_ylabel('Accélération (x)')\n",
    "    ax2.set_title('Accélération vs Poids')\n",
    "    ax2.grid(True, alpha=0.3)\n",
    "    \n",
    "    plt.tight_layout()\n",
    "    plt.show()\n",
    "    \n",
    "    return weights, costs, nodes\n",
    "\n",
    "weights, costs, nodes = weighted_astar_demo()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {
    "id": "conclusion"
   },
   "source": [
    "## Conclusion\n",
    "\n",
    "### Résumé des résultats\n",
    "\n",
    "1. **A*** : Optimal et efficace, meilleur compromis\n",
    "2. **UCS** : Optimal mais explore inutilement\n",
    "3. **Greedy** : Rapide mais sous-optimal\n",
    "4. **Incertitude ε** : Réduit significativement P(goal)\n",
    "5. **Heuristiques** : Manhattan réduit de 62% les nœuds développés\n",
    "6. **Weighted A*** : w=2.0 offre un bon compromis\n",
    "\n",
    "### Validation\n",
    "\n",
    "Les résultats théoriques (chaîne de Markov) concordent avec les simulations empiriques, validant notre modélisation.\n",
    "\n",
    "### Perspectives\n",
    "\n",
    "- Replanification dynamique\n",
    "- Politique optimale par MDP\n",
    "- Heuristiques apprises\n",
    "- Variantes mémoire (IDA*, SMA*)"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "codemirror_mode": {
    "name": "ipython",
    "version": 3
   },
   "file_extension": ".py",
   "mimetype": "text/x-python",
   "name": "python",
   "nbconvert_exporter": "python",
   "pygments_lexer": "ipython3",
   "version": "3.9.0"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 4
}

NameError: name 'null' is not defined